In [1]:
!pip install nbimporter


In [2]:
import nbimporter


ModuleNotFoundError: No module named 'nbimporter'

In [ ]:

import osmnx as ox
import networkx as nx
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import joblib


In [ ]:

# Files must be in same folder:
# - aqi_model.pkl
# - feature_order.pkl
# - delhi_ncr_area_wise_aqi_engineered.csv

model = joblib.load("aqi_rf_delhi_ncr.pkl")
feature_order = joblib.load("feature_orderv6.pkl")

df = pd.read_csv("delhi_ncr_area_wise_aqi_engineered.csv")
area_feature_table = df.groupby("Area").mean(numeric_only=True)

print("Model and data loaded")


In [ ]:

aqi_cache = {}

def predict_next_day_aqi(area):
    if area in aqi_cache:
        return aqi_cache[area]

    features = area_feature_table.loc[area].to_dict()
    tomorrow = datetime.now() + timedelta(days=1)
    features["Month"] = tomorrow.month

    X = pd.DataFrame([features]).reindex(columns=feature_order, fill_value=0)
    pred = float(model.predict(X)[0])

    aqi_cache[area] = pred
    return pred


In [ ]:
ox.settings.use_cache = True
ox.settings.log_console = False

G = ox.graph_from_place("Delhi, India", network_type="drive", simplify=True)

print(f"Road graph: {len(G.nodes)} nodes, {len(G.edges)} edges")

In [ ]:

AREA_COORDS = {
    "Rohini": (28.7499, 77.0565),
    "Pitampura": (28.7033, 77.1310),
    "Karol Bagh": (28.6517, 77.1906),
    "Connaught Place": (28.6315, 77.2167),
    "Janakpuri": (28.6219, 77.0878),
    "Saket": (28.5244, 77.2066),
    "Lajpat Nagar": (28.5677, 77.2433),
    "Faridabad": (28.4089, 77.3178),
}


In [ ]:

def nearest_node(area):
    lat, lon = AREA_COORDS[area]
    return ox.distance.nearest_nodes(G, lon, lat)


In [ ]:

def aqi_weight(u, v, data):
    length = data.get("length", 1.0)

    # approximate AQI exposure using average predicted AQI
    if not aqi_cache:
        avg_aqi = np.mean([predict_next_day_aqi(a) for a in AREA_COORDS])
    else:
        avg_aqi = np.mean(list(aqi_cache.values()))

    return length * (avg_aqi / 100.0)


In [ ]:

def fastest_route(start_area, end_area):
    s = nearest_node(start_area)
    e = nearest_node(end_area)
    return nx.shortest_path(G, s, e, weight="length")

def safest_route(start_area, end_area):
    s = nearest_node(start_area)
    e = nearest_node(end_area)
    return nx.shortest_path(G, s, e, weight=aqi_weight)


In [ ]:

def route_distance_km(path):
    distance = 0

    for u, v in zip(path[:-1], path[1:]):
        edge_data = G.get_edge_data(u, v)

        # if multiple edges exist, take the first one
        if isinstance(edge_data, dict):
            edge_data = list(edge_data.values())[0]

        distance += edge_data.get("length", 0)

    return round(distance / 1000, 2)



In [ ]:

start, end = "Rohini", "Faridabad"

fast = fastest_route(start, end)
safe = safest_route(start, end)

print("FASTEST route distance (km):", route_distance_km(fast))
print("SAFEST  route distance (km):", route_distance_km(safe))

print("Predicted AQI values used:")
for a in ["Rohini","Janakpuri","Saket","Lajpat Nagar","Faridabad"]:
    if a in area_feature_table.index:
        print(a, "→", round(predict_next_day_aqi(a),2))


In [ ]:
pip install folium


In [ ]:
import folium

In [ ]:
def route_to_gdf(G, route):
    # Extract edges for the route
    edges = []
    for u, v in zip(route[:-1], route[1:]):
        data = G.get_edge_data(u, v)
        if isinstance(data, dict):
            data = list(data.values())[0]
        geom = data.get("geometry")
        if geom is not None:
            edges.append(geom)
    return edges


In [ ]:
# Center map roughly between start & end
start_area, end_area = "Rohini", "Faridabad"
start_node = nearest_node(start_area)
end_node = nearest_node(end_area)

center_lat = (G.nodes[start_node]["y"] + G.nodes[end_node]["y"]) / 2
center_lon = (G.nodes[start_node]["x"] + G.nodes[end_node]["x"]) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=11)


In [ ]:
# FASTEST route (red)
fast_route = fastest_route(start_area, end_area)
fast_geoms = route_to_gdf(G, fast_route)
for geom in fast_geoms:
    folium.GeoJson(
        geom,
        style_function=lambda x: {"color": "red", "weight": 5, "opacity": 0.8},
        name="Fastest Route"
    ).add_to(m)

# SAFEST route (green)
safe_route = safest_route(start_area, end_area)
safe_geoms = route_to_gdf(G, safe_route)
for geom in safe_geoms:
    folium.GeoJson(
        geom,
        style_function=lambda x: {"color": "green", "weight": 5, "opacity": 0.8},
        name="Safest Route"
    ).add_to(m)

folium.LayerControl().add_to(m)
m

In [ ]:
# Start & End markers
s_lat, s_lon = AREA_COORDS[start_area]
e_lat, e_lon = AREA_COORDS[end_area]

folium.Marker([s_lat, s_lon], popup=f"Start: {start_area}", icon=folium.Icon(color="blue")).add_to(m)
folium.Marker([e_lat, e_lon], popup=f"End: {end_area}", icon=folium.Icon(color="black")).add_to(m)

m